In [1]:
import pandas as pd
import epi_utils as eu

In [9]:
from tqdm.auto import tqdm

In [10]:
tqdm.pandas()

## Loading all chr1 data

In [3]:
# Loading histone
histone_df = pd.read_csv("dataset/histone/chr1.csv")
display(histone_df)

,chrom,chromStart,chromEnd,name,length,type
0,chr1,713385,713531,chr1_378,146,h3k4me3
1,chr1,713593,713739,chr1_379,146,h3k4me3
2,chr1,713828,713974,chr1_380,146,h3k4me3
3,chr1,714270,714416,chr1_381,146,h3k4me3
4,chr1,714393,714539,chr1_382,146,h3k4me3
...,...,...,...,...,...,...
30368,chr1,248834918,248835064,chr1_1128451,146,h3k9me3
30369,chr1,248875103,248875249,chr1_1128652,146,h3k9me3
30370,chr1,248890138,248890284,chr1_1128720,146,h3k9me3
30371,chr1,249072368,249072514,chr1_1128874,146,h3k9me3


In [4]:
# Loading NCBI Refseq

refseq_df = pd.read_csv("dataset/ncbiRefSeq/merged/chr1.csv")
display(refseq_df.head())

,Chromosome,Source,Feature,Start,End,Score,Strand,Frame,gene_id,transcript_id,gene_name,exon_number,exon_id,tss
0,chr1,ncbiRefSeq.2021-05-17,transcript,249230769,249231298,.,+,.,RPL23AP25,RPL23AP25,RPL23AP25,NaN,NaN,249230769
1,chr1,ncbiRefSeq.2021-05-17,exon,249230769,249231298,.,+,.,RPL23AP25,RPL23AP25,RPL23AP25,1.0,RPL23AP25.1,249230769
2,chr1,ncbiRefSeq.2021-05-17,transcript,249206890,249206992,.,-,.,RNU6-1205P,RNU6-1205P,RNU6-1205P,NaN,NaN,249206992
3,chr1,ncbiRefSeq.2021-05-17,exon,249206890,249206992,.,-,.,RNU6-1205P,RNU6-1205P,RNU6-1205P,1.0,RNU6-1205P.1,249206992
4,chr1,ncbiRefSeq.2021-05-17,transcript,249200433,249213345,.,+,.,PGBD2,NM_001017434.2,PGBD2,NaN,NaN,249200433


In [8]:
def count_histone(histone_df, row, threshold = 0.8, histone_length = 146):
    tss = row["tss"]
    hist_df = histone_df.loc[# Inside the +/- 2k from TSS
                             (((histone_df["chromStart"] >= tss - 2000) & (histone_df["chromEnd"] <= tss + 2000)) |
                             # Intersect with -2k or +2 from TSS
                             (((histone_df["chromEnd"] - (tss - 2000))/histone_length).between(threshold, 1.0)) |
                             ((((tss + 2000) - histone_df["chromStart"])/histone_length).between(threshold, 1.0)))]
    
    return len(hist_df.index)

# Counting Histone

In [26]:
histone_list = ["h3k4me3", "h3k9ac", "h3k9me3", "h3k27ac", "h3k27me3"]

In [23]:
def count_histone_by_type(refseq_df, histone_df, histone_type):
    histone_type_df = histone_df[histone_df['type'] == histone_type]
    refseq_df.loc[:, histone_type] = refseq_df.progress_apply(\
                                lambda row: eu.count_histone(histone_type_df, row), axis=1)
    
    return refseq_df

## H3K4me3

In [24]:
refseq_df = count_histone_by_type(refseq_df, histone_df, "h3k4me3")

  0%|          | 0/193194 [00:00<?, ?it/s]

In [25]:
display(refseq_df.head())

,Chromosome,Source,Feature,Start,End,Score,Strand,Frame,gene_id,transcript_id,gene_name,exon_number,exon_id,tss,h3k4me3,h3k9ac
0,chr1,ncbiRefSeq.2021-05-17,transcript,249230769,249231298,.,+,.,RPL23AP25,RPL23AP25,RPL23AP25,NaN,NaN,249230769,0,0
1,chr1,ncbiRefSeq.2021-05-17,exon,249230769,249231298,.,+,.,RPL23AP25,RPL23AP25,RPL23AP25,1.0,RPL23AP25.1,249230769,0,0
2,chr1,ncbiRefSeq.2021-05-17,transcript,249206890,249206992,.,-,.,RNU6-1205P,RNU6-1205P,RNU6-1205P,NaN,NaN,249206992,0,0
3,chr1,ncbiRefSeq.2021-05-17,exon,249206890,249206992,.,-,.,RNU6-1205P,RNU6-1205P,RNU6-1205P,1.0,RNU6-1205P.1,249206992,0,0
4,chr1,ncbiRefSeq.2021-05-17,transcript,249200433,249213345,.,+,.,PGBD2,NM_001017434.2,PGBD2,NaN,NaN,249200433,5,4


## H3K9ac

In [27]:
refseq_df = count_histone_by_type(refseq_df, histone_df, "h3k9ac")
display(refseq_df.head())

  0%|          | 0/193194 [00:00<?, ?it/s]

,Chromosome,Source,Feature,Start,End,Score,Strand,Frame,gene_id,transcript_id,gene_name,exon_number,exon_id,tss,h3k4me3,h3k9ac
0,chr1,ncbiRefSeq.2021-05-17,transcript,249230769,249231298,.,+,.,RPL23AP25,RPL23AP25,RPL23AP25,NaN,NaN,249230769,0,0
1,chr1,ncbiRefSeq.2021-05-17,exon,249230769,249231298,.,+,.,RPL23AP25,RPL23AP25,RPL23AP25,1.0,RPL23AP25.1,249230769,0,0
2,chr1,ncbiRefSeq.2021-05-17,transcript,249206890,249206992,.,-,.,RNU6-1205P,RNU6-1205P,RNU6-1205P,NaN,NaN,249206992,0,0
3,chr1,ncbiRefSeq.2021-05-17,exon,249206890,249206992,.,-,.,RNU6-1205P,RNU6-1205P,RNU6-1205P,1.0,RNU6-1205P.1,249206992,0,0
4,chr1,ncbiRefSeq.2021-05-17,transcript,249200433,249213345,.,+,.,PGBD2,NM_001017434.2,PGBD2,NaN,NaN,249200433,5,4


## H3K9me3

In [28]:
refseq_df = count_histone_by_type(refseq_df, histone_df, "h3k9me3")
display(refseq_df.head())

  0%|          | 0/193194 [00:00<?, ?it/s]

,Chromosome,Source,Feature,Start,End,Score,Strand,Frame,gene_id,transcript_id,gene_name,exon_number,exon_id,tss,h3k4me3,h3k9ac,h3k9me3
0,chr1,ncbiRefSeq.2021-05-17,transcript,249230769,249231298,.,+,.,RPL23AP25,RPL23AP25,RPL23AP25,NaN,NaN,249230769,0,0,0
1,chr1,ncbiRefSeq.2021-05-17,exon,249230769,249231298,.,+,.,RPL23AP25,RPL23AP25,RPL23AP25,1.0,RPL23AP25.1,249230769,0,0,0
2,chr1,ncbiRefSeq.2021-05-17,transcript,249206890,249206992,.,-,.,RNU6-1205P,RNU6-1205P,RNU6-1205P,NaN,NaN,249206992,0,0,0
3,chr1,ncbiRefSeq.2021-05-17,exon,249206890,249206992,.,-,.,RNU6-1205P,RNU6-1205P,RNU6-1205P,1.0,RNU6-1205P.1,249206992,0,0,0
4,chr1,ncbiRefSeq.2021-05-17,transcript,249200433,249213345,.,+,.,PGBD2,NM_001017434.2,PGBD2,NaN,NaN,249200433,5,4,0


## H3K27ac

In [29]:
refseq_df = count_histone_by_type(refseq_df, histone_df, "h3k27ac")
display(refseq_df.head())

  0%|          | 0/193194 [00:00<?, ?it/s]

,Chromosome,Source,Feature,Start,End,Score,Strand,Frame,gene_id,transcript_id,gene_name,exon_number,exon_id,tss,h3k4me3,h3k9ac,h3k9me3,h3k27ac
0,chr1,ncbiRefSeq.2021-05-17,transcript,249230769,249231298,.,+,.,RPL23AP25,RPL23AP25,RPL23AP25,NaN,NaN,249230769,0,0,0,0
1,chr1,ncbiRefSeq.2021-05-17,exon,249230769,249231298,.,+,.,RPL23AP25,RPL23AP25,RPL23AP25,1.0,RPL23AP25.1,249230769,0,0,0,0
2,chr1,ncbiRefSeq.2021-05-17,transcript,249206890,249206992,.,-,.,RNU6-1205P,RNU6-1205P,RNU6-1205P,NaN,NaN,249206992,0,0,0,0
3,chr1,ncbiRefSeq.2021-05-17,exon,249206890,249206992,.,-,.,RNU6-1205P,RNU6-1205P,RNU6-1205P,1.0,RNU6-1205P.1,249206992,0,0,0,0
4,chr1,ncbiRefSeq.2021-05-17,transcript,249200433,249213345,.,+,.,PGBD2,NM_001017434.2,PGBD2,NaN,NaN,249200433,5,4,0,4


## H3K27me3

In [30]:
refseq_df = count_histone_by_type(refseq_df, histone_df, "h3k27me3")
display(refseq_df.head())

  0%|          | 0/193194 [00:00<?, ?it/s]

,Chromosome,Source,Feature,Start,End,Score,Strand,Frame,gene_id,transcript_id,gene_name,exon_number,exon_id,tss,h3k4me3,h3k9ac,h3k9me3,h3k27ac,h3k27me3
0,chr1,ncbiRefSeq.2021-05-17,transcript,249230769,249231298,.,+,.,RPL23AP25,RPL23AP25,RPL23AP25,NaN,NaN,249230769,0,0,0,0,0
1,chr1,ncbiRefSeq.2021-05-17,exon,249230769,249231298,.,+,.,RPL23AP25,RPL23AP25,RPL23AP25,1.0,RPL23AP25.1,249230769,0,0,0,0,0
2,chr1,ncbiRefSeq.2021-05-17,transcript,249206890,249206992,.,-,.,RNU6-1205P,RNU6-1205P,RNU6-1205P,NaN,NaN,249206992,0,0,0,0,0
3,chr1,ncbiRefSeq.2021-05-17,exon,249206890,249206992,.,-,.,RNU6-1205P,RNU6-1205P,RNU6-1205P,1.0,RNU6-1205P.1,249206992,0,0,0,0,0
4,chr1,ncbiRefSeq.2021-05-17,transcript,249200433,249213345,.,+,.,PGBD2,NM_001017434.2,PGBD2,NaN,NaN,249200433,5,4,0,4,0


## Total

In [35]:
# h3k4me3	h3k9ac	h3k9me3	h3k27ac	h3k27me3
refseq_df.loc[:, "histone_count_total"] =   refseq_df["h3k4me3"] + \
                                            refseq_df["h3k9ac"] + \
                                            refseq_df["h3k9me3"] + \
                                            refseq_df["h3k27ac"] + \
                                            refseq_df["h3k27me3"]

In [37]:
display(refseq_df.head())

,Chromosome,Source,Feature,Start,End,Score,Strand,Frame,gene_id,transcript_id,gene_name,exon_number,exon_id,tss,h3k4me3,h3k9ac,h3k9me3,h3k27ac,h3k27me3,histone_count_total
0,chr1,ncbiRefSeq.2021-05-17,transcript,249230769,249231298,.,+,.,RPL23AP25,RPL23AP25,RPL23AP25,NaN,NaN,249230769,0,0,0,0,0,0
1,chr1,ncbiRefSeq.2021-05-17,exon,249230769,249231298,.,+,.,RPL23AP25,RPL23AP25,RPL23AP25,1.0,RPL23AP25.1,249230769,0,0,0,0,0,0
2,chr1,ncbiRefSeq.2021-05-17,transcript,249206890,249206992,.,-,.,RNU6-1205P,RNU6-1205P,RNU6-1205P,NaN,NaN,249206992,0,0,0,0,0,0
3,chr1,ncbiRefSeq.2021-05-17,exon,249206890,249206992,.,-,.,RNU6-1205P,RNU6-1205P,RNU6-1205P,1.0,RNU6-1205P.1,249206992,0,0,0,0,0,0
4,chr1,ncbiRefSeq.2021-05-17,transcript,249200433,249213345,.,+,.,PGBD2,NM_001017434.2,PGBD2,NaN,NaN,249200433,5,4,0,4,0,13


In [38]:
refseq_df.to_csv("dataset/histone_count/chr1.csv", header=True, index=False)

In [39]:
refseq_df.head(1000).to_csv("dataset/histone_count/chr1_1000.csv", header=True, index=False)

In [43]:
refseq_df[refseq_df['h3k9ac'] != refseq_df['h3k27ac']].head()

,Chromosome,Source,Feature,Start,End,Score,Strand,Frame,gene_id,transcript_id,gene_name,exon_number,exon_id,tss,h3k4me3,h3k9ac,h3k9me3,h3k27ac,h3k27me3,histone_count_total
174,chr1,ncbiRefSeq.2021-05-17,transcript,249132422,249143716,.,+,.,ZNF672,NM_024836.3,ZNF672,NaN,NaN,249132422,4,3,0,2,0,9
175,chr1,ncbiRefSeq.2021-05-17,exon,249132422,249132750,.,+,.,ZNF672,NM_024836.3,ZNF672,1.0,NM_024836.3.1,249132422,4,3,0,2,0,9
176,chr1,ncbiRefSeq.2021-05-17,5UTR,249132422,249132750,.,+,.,ZNF672,NM_024836.3,ZNF672,1.0,NM_024836.3.1,249132422,4,3,0,2,0,9
187,chr1,ncbiRefSeq.2021-05-17,transcript,249120575,249120642,.,+,.,MIR3124,NR_036070.1,MIR3124,NaN,NaN,249120575,5,5,0,4,0,14
188,chr1,ncbiRefSeq.2021-05-17,exon,249120575,249120642,.,+,.,MIR3124,NR_036070.1,MIR3124,1.0,NR_036070.1.1,249120575,5,5,0,4,0,14


# HepG2 that contains TSS

In [106]:
hepg2 = pd.read_csv("dataset/HepG2/merged/chr1.csv")
hepg2.head()

,test_id,gene_id,gene,locus,sample_1,sample_2,status,value_1,value_2,log2(fold_change),test_stat,p_value,q_value,significant,chrom,chromStart,chromEnd
0,XLOC_000001,XLOC_000001,OR4F5,chr1:69090-70008,hepg2_hr2,hepg2_hr3,NOTEST,0.000000,0.000000,0.000000,0.000000,1.00000,1.000000,no,chr1,69090,70008
1,XLOC_000002,XLOC_000002,"LOC100132062,LOC100133331",chr1:323891-328581,hepg2_hr2,hepg2_hr3,NOTEST,0.047392,0.034159,-0.472389,0.000000,1.00000,1.000000,no,chr1,323891,328581
2,XLOC_000003,XLOC_000003,OR4F29,chr1:367658-368597,hepg2_hr2,hepg2_hr3,NOTEST,0.000000,0.000000,0.000000,0.000000,1.00000,1.000000,no,chr1,367658,368597
3,XLOC_000004,XLOC_000004,LOC643837,chr1:761585-794889,hepg2_hr2,hepg2_hr3,OK,2.468570,2.805800,0.184736,0.455074,0.63585,0.999565,no,chr1,761585,794889
4,XLOC_000005,XLOC_000005,-,chr1:840263-843900,hepg2_hr2,hepg2_hr3,NOTEST,0.000000,0.000000,0.000000,0.000000,1.00000,1.000000,no,chr1,840263,843900


In [107]:
hepg2.shape

(2868, 17)

In [153]:
hepg2_refseq = refseq_df[(refseq_df['tss'] >= 851135) & (refseq_df['tss'] <= 917473)]
display(hepg2_refseq[hepg2_refseq["histone_count_total"] > 0])
display(hepg2_refseq.sum(numeric_only=True).to_dict())

,Chromosome,Source,Feature,Start,End,Score,Strand,Frame,gene_id,transcript_id,gene_name,exon_number,exon_id,tss,h3k4me3,h3k9ac,h3k9me3,h3k27ac,h3k27me3,histone_count_total
179092,chr1,ncbiRefSeq.2021-05-17,transcript,910577,916553,.,-,.,PERM1,NM_001369898.1,PERM1,NaN,NaN,916553,0,0,0,0,1,1
179098,chr1,ncbiRefSeq.2021-05-17,exon,914260,916553,.,-,.,PERM1,NM_001369898.1,PERM1,1.0,NM_001369898.1.1,916553,0,0,0,0,1,1
179099,chr1,ncbiRefSeq.2021-05-17,CDS,914260,916409,.,-,0,PERM1,NM_001369898.1,PERM1,1.0,NM_001369898.1.1,916409,0,0,0,0,1,1
179100,chr1,ncbiRefSeq.2021-05-17,5UTR,916409,916553,.,-,.,PERM1,NM_001369898.1,PERM1,1.0,NM_001369898.1.1,916553,0,0,0,0,1,1
179101,chr1,ncbiRefSeq.2021-05-17,start_codon,916406,916409,.,-,0,PERM1,NM_001369898.1,PERM1,1.0,NM_001369898.1.1,916409,0,0,0,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
191889,chr1_jh636053_fix,ncbiRefSeq.2021-05-17,CDS,902871,902923,.,+,2,NBPF26,NM_001351372.1,NBPF26,21.0,NM_001351372.1.21,902871,3,2,0,3,0,8
191890,chr1_jh636053_fix,ncbiRefSeq.2021-05-17,exon,903561,903734,.,+,.,NBPF26,NM_001351372.1,NBPF26,22.0,NM_001351372.1.22,903561,3,2,0,3,0,8
191891,chr1_jh636053_fix,ncbiRefSeq.2021-05-17,CDS,903561,903734,.,+,1,NBPF26,NM_001351372.1,NBPF26,22.0,NM_001351372.1.22,903561,3,2,0,3,0,8
191892,chr1_jh636053_fix,ncbiRefSeq.2021-05-17,exon,904448,904557,.,+,.,NBPF26,NM_001351372.1,NBPF26,23.0,NM_001351372.1.23,904448,1,0,0,0,0,1


{'Start': 380192853.0,
 'End': 380472505.0,
 'exon_number': 3410.0,
 'tss': 380283998.0,
 'h3k4me3': 346.0,
 'h3k9ac': 298.0,
 'h3k9me3': 0.0,
 'h3k27ac': 283.0,
 'h3k27me3': 30.0,
 'histone_count_total': 957.0}

In [128]:
def hepg2_intersect_refseq(refseq_df, row):
    hepg2_refseq = refseq_df[(refseq_df['tss'] >= row['chromStart']) & (refseq_df['tss'] <= row['chromEnd'])]
    hepg2_dict = hepg2_refseq.sum(numeric_only=True).to_dict()

    if (len(hepg2_refseq) == 0):
        hepg2_dict.update({'status_refseq': 0})
    else:
      hepg2_dict.update({'status_refseq': 1})
    
    return hepg2_dict

In [129]:
hepg2.loc[:, 'hepg2_refseq'] = hepg2.progress_apply(lambda row: hepg2_intersect_refseq(refseq_df, row), axis = 1)

  0%|          | 0/2868 [00:00<?, ?it/s]

In [130]:
hepg2.head()

,test_id,gene_id,gene,locus,sample_1,sample_2,status,value_1,value_2,log2(fold_change),test_stat,p_value,q_value,significant,chrom,chromStart,chromEnd,hepg2_refseq
0,XLOC_000001,XLOC_000001,OR4F5,chr1:69090-70008,hepg2_hr2,hepg2_hr3,NOTEST,0.000000,0.000000,0.000000,0.000000,1.00000,1.000000,no,chr1,69090,70008,"{'Start': 1807213.0, 'End': 1811137.0, 'exon_n..."
1,XLOC_000002,XLOC_000002,"LOC100132062,LOC100133331",chr1:323891-328581,hepg2_hr2,hepg2_hr3,NOTEST,0.047392,0.034159,-0.472389,0.000000,1.00000,1.000000,no,chr1,323891,328581,"{'Start': 1620441.0, 'End': 1629569.0, 'exon_n..."
2,XLOC_000003,XLOC_000003,OR4F29,chr1:367658-368597,hepg2_hr2,hepg2_hr3,NOTEST,0.000000,0.000000,0.000000,0.000000,1.00000,1.000000,no,chr1,367658,368597,"{'Start': 1839226.0, 'End': 1842046.0, 'exon_n..."
3,XLOC_000004,XLOC_000004,LOC643837,chr1:761585-794889,hepg2_hr2,hepg2_hr3,OK,2.468570,2.805800,0.184736,0.455074,0.63585,0.999565,no,chr1,761585,794889,"{'Start': 85623432.0, 'End': 86049681.0, 'exon..."
4,XLOC_000005,XLOC_000005,-,chr1:840263-843900,hepg2_hr2,hepg2_hr3,NOTEST,0.000000,0.000000,0.000000,0.000000,1.00000,1.000000,no,chr1,840263,843900,"{'Start': 0.0, 'End': 0.0, 'exon_number': 0.0,..."


In [131]:
hepg2_refseq_df = pd.json_normalize(hepg2['hepg2_refseq'])

In [132]:
hepg2_refseq_df.head()

,Start,End,exon_number,tss,h3k4me3,h3k9ac,h3k9me3,h3k27ac,h3k27me3,histone_count_total,status_refseq
0,1807213.0,1811137.0,240.0,1809557.0,0.0,0.0,0.0,0.0,0.0,0.0,1
1,1620441.0,1629569.0,24.0,1620509.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2,1839226.0,1842046.0,4.0,1839226.0,0.0,0.0,0.0,0.0,0.0,0.0,1
3,85623432.0,86049681.0,259.0,85741243.0,0.0,0.0,0.0,0.0,0.0,0.0,1
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [133]:
hepg2_histone = pd.concat([hepg2, hepg2_refseq_df], axis = 1)

In [137]:
hepg2_histone = hepg2_histone[hepg2_histone['status_refseq'] == 1]

In [141]:
hepg2_histone = hepg2_histone[['test_id', 'gene_id', 'gene', 'locus', 'sample_1', 'sample_2', 'status',
       'value_1', 'value_2', 'log2(fold_change)', 'test_stat', 'p_value',
       'q_value', 'significant', 'chrom', 'chromStart', 'chromEnd',
       'h3k4me3', 'h3k9ac', 'h3k9me3', 'h3k27ac', 'h3k27me3', 'histone_count_total']]

In [143]:
hepg2_histone = hepg2_histone.astype({'h3k4me3': 'int32', 'h3k9ac':'int32', 'h3k9me3':'int32', 'h3k27ac':'int32', 'h3k27me3':'int32', 'histone_count_total':'int32'})

In [144]:
hepg2_histone

,test_id,gene_id,gene,locus,sample_1,sample_2,status,value_1,value_2,log2(fold_change),...,significant,chrom,chromStart,chromEnd,h3k4me3,h3k9ac,h3k9me3,h3k27ac,h3k27me3,histone_count_total
0,XLOC_000001,XLOC_000001,OR4F5,chr1:69090-70008,hepg2_hr2,hepg2_hr3,NOTEST,0.000000,0.000000,0.000000,...,no,chr1,69090,70008,0,0,0,0,0,0
1,XLOC_000002,XLOC_000002,"LOC100132062,LOC100133331",chr1:323891-328581,hepg2_hr2,hepg2_hr3,NOTEST,0.047392,0.034159,-0.472389,...,no,chr1,323891,328581,0,0,0,0,0,0
2,XLOC_000003,XLOC_000003,OR4F29,chr1:367658-368597,hepg2_hr2,hepg2_hr3,NOTEST,0.000000,0.000000,0.000000,...,no,chr1,367658,368597,0,0,0,0,0,0
3,XLOC_000004,XLOC_000004,LOC643837,chr1:761585-794889,hepg2_hr2,hepg2_hr3,OK,2.468570,2.805800,0.184736,...,no,chr1,761585,794889,0,0,0,0,0,0
5,XLOC_000006,XLOC_000006,SAMD11,chr1:851135-917473,hepg2_hr2,hepg2_hr3,NOTEST,0.088845,0.136965,0.624439,...,no,chr1,851135,917473,346,298,0,283,30,957
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2825,XLOC_002826,XLOC_002826,-,chr1:231010111-231014583,hepg2_hr2,hepg2_hr3,OK,0.354983,0.599736,0.756576,...,no,chr1,231010111,231014583,0,0,0,0,0,0
2851,XLOC_002852,XLOC_002852,-,chr1:242079315-242080039,hepg2_hr2,hepg2_hr3,NOTEST,0.073273,0.085609,0.224481,...,no,chr1,242079315,242080039,0,0,0,0,0,0
2863,XLOC_002864,XLOC_002864,-,chr1:247363335-247374187,hepg2_hr2,hepg2_hr3,OK,0.398402,0.602551,0.596858,...,no,chr1,247363335,247374187,0,0,0,0,0,0
2864,XLOC_002865,XLOC_002865,-,chr1:247363335-247374187,hepg2_hr2,hepg2_hr3,OK,0.762831,0.705872,-0.111956,...,no,chr1,247363335,247374187,0,0,0,0,0,0


In [145]:
hepg2_histone.to_csv("dataset/HepG2/hepg2_with_histone.csv", header=True, index=False)